# Fake News Detection tiếng Việt bằng SVM trên 2 tập dữ liệu VFND

Notebook này sử dụng 2 file dữ liệu trong repo **VFND - Vietnamese Fake News Dataset**:

1. `vn_news_226_tlfr.csv`
2. `vn_news_223_tdlfr.csv`

## Hướng làm chính trong notebook

Do bộ dữ liệu nhỏ, ta làm theo hướng:

1. Đọc cả 2 tập dữ liệu.
2. Gộp 2 tập dữ liệu lại.
3. Chuẩn hóa nhãn:
   - `0`: Real - Tin thật
   - `1`: Fake - Tin giả
4. Tiền xử lý văn bản tiếng Việt.
5. Xóa bản ghi trùng lặp.
6. Chia train/test theo tỷ lệ 80/20.
7. Huấn luyện mô hình chính: **SVM tuyến tính**.
8. So sánh SVM với:
   - Naive Bayes
   - Logistic Regression
9. Thực nghiệm phụ:
   - Train trên `vn_news_226_tlfr.csv`
   - Test trên `vn_news_223_tdlfr.csv`

## Mục tiêu bài toán

Phân loại một tin tiếng Việt là:

- **Tin thật**
- **Tin giả**

## 1. Cài đặt và import thư viện

In [ ]:
# Nếu thiếu thư viện, bỏ dấu # ở dòng dưới rồi chạy
# !pip install pandas numpy scikit-learn matplotlib seaborn joblib

import os
import re
import csv
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from io import StringIO
from urllib.request import urlopen

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

RANDOM_STATE = 42

plt.rcParams["figure.figsize"] = (8, 5)
sns.set_theme(style="whitegrid")

## 2. Khai báo đường dẫn 2 tập dữ liệu

Notebook mặc định tải trực tiếp từ GitHub raw.

Nếu muốn chạy offline, tải 2 file CSV về cùng thư mục với notebook rồi đổi đường dẫn thành tên file local.

In [ ]:
RAW_URL_226 = "https://raw.githubusercontent.com/WhySchools/VFND-vietnamese-fake-news-datasets/master/CSV/vn_news_226_tlfr.csv"
RAW_URL_223 = "https://raw.githubusercontent.com/WhySchools/VFND-vietnamese-fake-news-datasets/master/CSV/vn_news_223_tdlfr.csv"

DATA_SOURCES = {
    "vn_news_226_tlfr": RAW_URL_226,
    "vn_news_223_tdlfr": RAW_URL_223
}

# Nếu chạy offline, dùng dạng này:
# DATA_SOURCES = {
#     "vn_news_226_tlfr": "vn_news_226_tlfr.csv",
#     "vn_news_223_tdlfr": "vn_news_223_tdlfr.csv"
# }

## 3. Hàm đọc file CSV của VFND

Một số file chứa văn bản dài, dấu phẩy hoặc dấu xuống dòng trong nội dung tin. Vì vậy, hàm dưới đây đọc dữ liệu theo cách linh hoạt hơn.

In [ ]:
def load_vfnd_csv(source):
    """
    Đọc file CSV của VFND từ URL hoặc file local.

    Kết quả trả về DataFrame có 2 cột chính:
    - text
    - label
    """
    # Cách 1: thử đọc trực tiếp bằng pandas
    try:
        df = pd.read_csv(source, encoding="utf-8", engine="python")
        df.columns = [str(c).strip().lower() for c in df.columns]

        # Chuẩn hóa tên cột có thể gặp
        rename_map = {}
        for col in df.columns:
            if col.lower() in ["text", "content", "article", "body", "news"]:
                rename_map[col] = "text"
            if col.lower() in ["label", "target", "class"]:
                rename_map[col] = "label"

        df = df.rename(columns=rename_map)

        if "text" in df.columns and "label" in df.columns:
            return df[["text", "label"]].copy()
    except Exception as e:
        print("Không đọc được trực tiếp bằng pandas, chuyển sang cách đọc thủ công.")
        print("Lỗi:", e)

    # Cách 2: đọc thủ công
    if str(source).startswith("http"):
        raw_text = urlopen(source).read().decode("utf-8", errors="replace")
    else:
        with open(source, "r", encoding="utf-8", errors="replace") as f:
            raw_text = f.read()

    csv_reader = csv.reader(
        StringIO(raw_text),
        delimiter=",",
        quotechar='"',
        skipinitialspace=True
    )

    rows = [row for row in csv_reader if len(row) > 0]

    if not rows:
        raise ValueError("File rỗng hoặc không đọc được dữ liệu.")

    header = [x.strip().lower() for x in rows[0]]

    if "text" in header and "label" in header:
        text_idx = header.index("text")
        label_idx = header.index("label")
        data_rows = rows[1:]
    else:
        # Nếu không có header rõ ràng, giả định cột đầu là text, cột cuối là label
        text_idx = 0
        label_idx = -1
        data_rows = rows

    data = []
    for row in data_rows:
        if len(row) < 2:
            continue

        text = row[text_idx].strip()
        label = row[label_idx].strip()

        if text != "" and label != "":
            data.append({
                "text": text,
                "label": label
            })

    return pd.DataFrame(data)

## 4. Đọc và gộp 2 tập dữ liệu

Ta thêm cột `source` để biết bản ghi đến từ file nào.

In [ ]:
dfs = []

for source_name, source_path in DATA_SOURCES.items():
    temp_df = load_vfnd_csv(source_path)
    temp_df["source"] = source_name
    dfs.append(temp_df)

df_raw = pd.concat(dfs, ignore_index=True)

print("Kích thước dữ liệu sau khi gộp thô:", df_raw.shape)
print("\nSố lượng mẫu theo từng nguồn:")
print(df_raw["source"].value_counts())

display(df_raw.head())

## 5. Kiểm tra dữ liệu ban đầu

In [ ]:
print("Thông tin dữ liệu:")
display(df_raw.info())

print("\nSố lượng giá trị thiếu:")
print(df_raw.isna().sum())

print("\nPhân bố nhãn ban đầu:")
print(df_raw["label"].value_counts(dropna=False))

print("\nMột vài dòng dữ liệu:")
display(df_raw.sample(min(5, len(df_raw)), random_state=RANDOM_STATE))

## 6. Chuẩn hóa nhãn

Theo quy ước của VFND:

- `0`: Real - Tin thật
- `1`: Fake - Tin giả

In [ ]:
def normalize_label(label):
    """
    Chuẩn hóa nhãn về dạng số:
    - 0: Real - Tin thật
    - 1: Fake - Tin giả
    """
    label = str(label).strip().lower()

    if label in ["0", "real", "true", "thật", "tin thật"]:
        return 0

    if label in ["1", "fake", "false", "giả", "tin giả"]:
        return 1

    # Xử lý trường hợp label là 0.0 hoặc 1.0
    try:
        value = int(float(label))
        if value in [0, 1]:
            return value
    except:
        pass

    return np.nan


label_names = {
    0: "Real - Tin thật",
    1: "Fake - Tin giả"
}

df = df_raw.copy()
df["label"] = df["label"].apply(normalize_label)

df = df.dropna(subset=["text", "label"])
df["label"] = df["label"].astype(int)

print("Kích thước sau chuẩn hóa nhãn:", df.shape)
print(df["label"].value_counts().sort_index().rename(index=label_names))

## 7. Tiền xử lý văn bản tiếng Việt

Các bước tiền xử lý:

1. Chuyển chữ thường.
2. Xóa URL.
3. Xóa email.
4. Xóa số.
5. Xóa ký tự đặc biệt.
6. Xóa khoảng trắng thừa.
7. Loại bỏ stopword tiếng Việt cơ bản.

Lưu ý: Nếu muốn nâng cao, có thể dùng thư viện `underthesea` hoặc `pyvi` để tách từ tiếng Việt.

In [ ]:
vietnamese_stopwords = set([
    "và", "là", "của", "có", "cho", "với", "một", "các", "những",
    "được", "trong", "khi", "đã", "này", "đó", "thì", "mà", "ở",
    "từ", "về", "theo", "sau", "trước", "đến", "ra", "vào", "nên",
    "như", "trên", "dưới", "bị", "cũng", "nhiều", "rất", "lại",
    "sẽ", "đang", "không", "người", "việc", "năm", "ngày", "nói",
    "biết", "làm", "đi", "để", "vì", "do", "tại", "nếu", "hay"
])

def clean_text(text):
    text = str(text).lower()

    # Xóa URL
    text = re.sub(r"http\S+|www\.\S+", " ", text)

    # Xóa email
    text = re.sub(r"\S+@\S+", " ", text)

    # Xóa số
    text = re.sub(r"\d+", " ", text)

    # Giữ chữ cái tiếng Việt và khoảng trắng
    text = re.sub(r"[^a-zA-ZÀ-ỹà-ỹ\s]", " ", text)

    # Xóa khoảng trắng thừa
    text = re.sub(r"\s+", " ", text).strip()

    tokens = text.split()
    tokens = [word for word in tokens if word not in vietnamese_stopwords]

    return " ".join(tokens)


df["text"] = df["text"].astype(str)
df["clean_text"] = df["text"].apply(clean_text)

df = df[df["clean_text"].str.len() > 0].copy()

print("Kích thước sau tiền xử lý văn bản:", df.shape)
display(df[["source", "text", "clean_text", "label"]].head())

## 8. Xóa dữ liệu trùng lặp sau khi gộp

Đây là bước rất quan trọng.

Nếu không xóa trùng, cùng một tin có thể xuất hiện ở cả train và test, làm kết quả đánh giá bị cao ảo.

In [ ]:
before_drop = len(df)

# Xóa các bản ghi trùng cả nội dung đã làm sạch và nhãn
df = df.drop_duplicates(subset=["clean_text", "label"]).reset_index(drop=True)

after_drop = len(df)

print("Số bản ghi trước khi xóa trùng:", before_drop)
print("Số bản ghi sau khi xóa trùng:", after_drop)
print("Số bản ghi bị loại:", before_drop - after_drop)

print("\nPhân bố nhãn sau khi xóa trùng:")
print(df["label"].value_counts().sort_index().rename(index=label_names))

## 9. Trực quan hóa phân bố nhãn sau khi gộp và xóa trùng

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="label")
plt.title("Phân bố nhãn sau khi gộp 2 dataset và xóa trùng")
plt.xlabel("Nhãn")
plt.ylabel("Số lượng")
plt.xticks([0, 1], ["Real", "Fake"])
plt.show()

plt.figure(figsize=(7, 4))
sns.countplot(data=df, x="source", hue="label")
plt.title("Phân bố nhãn theo nguồn dữ liệu")
plt.xlabel("Nguồn dữ liệu")
plt.ylabel("Số lượng")
plt.legend(title="Nhãn", labels=["Real", "Fake"])
plt.xticks(rotation=10)
plt.show()

# PHẦN A - Thực nghiệm chính: Gộp 2 dataset rồi chia train/test

Đây là hướng nên dùng làm kết quả chính trong báo cáo.

## 10. Chia dữ liệu train/test

Ta chia dữ liệu sau khi gộp theo tỷ lệ:

- 80% train
- 20% test

Dùng `stratify=y` để giữ tỷ lệ Real/Fake giữa train và test.

In [ ]:
X = df["clean_text"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Số mẫu train:", len(X_train))
print("Số mẫu test:", len(X_test))

print("\nPhân bố nhãn train:")
print(y_train.value_counts().sort_index().rename(index=label_names))

print("\nPhân bố nhãn test:")
print(y_test.value_counts().sort_index().rename(index=label_names))

## 11. Xây dựng mô hình SVM tuyến tính

Pipeline gồm:

1. `TfidfVectorizer`: chuyển văn bản thành vector TF-IDF.
2. `LinearSVC`: mô hình Support Vector Machine tuyến tính.

SVM tuyến tính thường phù hợp với dữ liệu văn bản vì vector TF-IDF có số chiều lớn và thưa.

In [ ]:
svm_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=20000,
        ngram_range=(1, 2),
        min_df=1,
        max_df=0.95,
        sublinear_tf=True
    )),
    ("svm", LinearSVC(
        C=1.0,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ))
])

svm_pipeline.fit(X_train, y_train)

y_pred_svm = svm_pipeline.predict(X_test)

print("Kết quả SVM ban đầu:")
print(classification_report(
    y_test,
    y_pred_svm,
    target_names=["Real - Tin thật", "Fake - Tin giả"],
    zero_division=0
))

## 12. Tối ưu mô hình SVM bằng GridSearchCV

Ta tìm bộ tham số tốt hơn cho SVM:

- `tfidf__max_features`: số đặc trưng TF-IDF tối đa.
- `tfidf__ngram_range`: dùng unigram hoặc unigram + bigram.
- `svm__C`: tham số điều chuẩn của SVM.

In [ ]:
param_grid = {
    "tfidf__max_features": [5000, 10000, 20000],
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__min_df": [1, 2],
    "svm__C": [0.1, 1, 10]
}

grid_search = GridSearchCV(
    estimator=svm_pipeline,
    param_grid=param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print("Tham số tốt nhất:")
print(grid_search.best_params_)

print("\nF1-score trung bình tốt nhất trên cross-validation:")
print(grid_search.best_score_)

best_svm_model = grid_search.best_estimator_

## 13. Đánh giá SVM sau tối ưu trên tập test

In [ ]:
y_pred_best_svm = best_svm_model.predict(X_test)

svm_result = {
    "Mô hình": "SVM tuyến tính",
    "Accuracy": accuracy_score(y_test, y_pred_best_svm),
    "Precision": precision_score(y_test, y_pred_best_svm, zero_division=0),
    "Recall": recall_score(y_test, y_pred_best_svm, zero_division=0),
    "F1-score": f1_score(y_test, y_pred_best_svm, zero_division=0)
}

print("Kết quả SVM sau tối ưu:")
for key, value in svm_result.items():
    print(key, ":", value)

print("\nBáo cáo phân loại chi tiết:")
print(classification_report(
    y_test,
    y_pred_best_svm,
    target_names=["Real - Tin thật", "Fake - Tin giả"],
    zero_division=0
))

## 14. Ma trận nhầm lẫn của SVM

In [ ]:
cm = confusion_matrix(y_test, y_pred_best_svm)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Real", "Fake"],
    yticklabels=["Real", "Fake"]
)
plt.title("Confusion Matrix - SVM sau tối ưu")
plt.xlabel("Nhãn dự đoán")
plt.ylabel("Nhãn thực tế")
plt.show()

## 15. So sánh SVM với các thuật toán khác

Ta so sánh SVM với:

1. Multinomial Naive Bayes
2. Logistic Regression

Tất cả mô hình đều sử dụng TF-IDF để biểu diễn văn bản.

In [ ]:
comparison_models = {
    "SVM tuyến tính": best_svm_model,

    "Naive Bayes": Pipeline([
        ("tfidf", TfidfVectorizer(
            max_features=20000,
            ngram_range=(1, 2),
            min_df=1,
            max_df=0.95,
            sublinear_tf=True
        )),
        ("nb", MultinomialNB())
    ]),

    "Logistic Regression": Pipeline([
        ("tfidf", TfidfVectorizer(
            max_features=20000,
            ngram_range=(1, 2),
            min_df=1,
            max_df=0.95,
            sublinear_tf=True
        )),
        ("lr", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=RANDOM_STATE
        ))
    ])
}

results = []

for model_name, model in comparison_models.items():
    print("=" * 80)
    print("Mô hình:", model_name)

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    result = {
        "Mô hình": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1-score": f1_score(y_test, y_pred, zero_division=0)
    }

    results.append(result)

    print(classification_report(
        y_test,
        y_pred,
        target_names=["Real - Tin thật", "Fake - Tin giả"],
        zero_division=0
    ))

results_df = pd.DataFrame(results).sort_values(by="F1-score", ascending=False)
display(results_df)

## 16. Biểu đồ so sánh các mô hình

In [ ]:
results_melted = results_df.melt(
    id_vars="Mô hình",
    value_vars=["Accuracy", "Precision", "Recall", "F1-score"],
    var_name="Chỉ số",
    value_name="Giá trị"
)

plt.figure(figsize=(10, 6))
sns.barplot(data=results_melted, x="Mô hình", y="Giá trị", hue="Chỉ số")
plt.title("So sánh SVM với các thuật toán khác")
plt.ylim(0, 1.05)
plt.xticks(rotation=15)
plt.legend(loc="lower right")
plt.show()

## 17. Cross-validation cho mô hình SVM tốt nhất

Cross-validation giúp đánh giá mô hình ổn định hơn thay vì chỉ dựa vào một lần chia train/test.

In [ ]:
cv_scores = cross_val_score(
    best_svm_model,
    X,
    y,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

print("F1-score từng fold:", cv_scores)
print("F1-score trung bình:", cv_scores.mean())
print("Độ lệch chuẩn:", cv_scores.std())

# PHẦN B - Thực nghiệm phụ: Train một dataset, test dataset còn lại

Thực nghiệm này dùng để kiểm tra khả năng tổng quát hóa của mô hình.

Ví dụ:

- Train: `vn_news_226_tlfr.csv`
- Test: `vn_news_223_tdlfr.csv`

Lưu ý: Nếu 2 tập có nhiều nội dung trùng nhau, ta vẫn cần xóa trùng giữa train và test.

## 18. Chuẩn bị riêng từng dataset

In [ ]:
def prepare_dataframe(input_df):
    """
    Chuẩn hóa nhãn, tiền xử lý văn bản và xóa trùng cho từng dataframe.
    """
    temp = input_df.copy()
    temp["label"] = temp["label"].apply(normalize_label)
    temp = temp.dropna(subset=["text", "label"])
    temp["label"] = temp["label"].astype(int)
    temp["text"] = temp["text"].astype(str)
    temp["clean_text"] = temp["text"].apply(clean_text)
    temp = temp[temp["clean_text"].str.len() > 0]
    temp = temp.drop_duplicates(subset=["clean_text", "label"]).reset_index(drop=True)
    return temp


df_226 = prepare_dataframe(df_raw[df_raw["source"] == "vn_news_226_tlfr"].copy())
df_223 = prepare_dataframe(df_raw[df_raw["source"] == "vn_news_223_tdlfr"].copy())

print("Dataset 226:", df_226.shape)
print(df_226["label"].value_counts().sort_index().rename(index=label_names))

print("\nDataset 223:", df_223.shape)
print(df_223["label"].value_counts().sort_index().rename(index=label_names))

## 19. Xóa các mẫu trùng giữa train dataset và test dataset

Nếu một mẫu xuất hiện ở cả 2 dataset, loại mẫu đó khỏi test để đánh giá công bằng hơn.

In [ ]:
train_df_external = df_226.copy()
test_df_external = df_223.copy()

train_text_set = set(train_df_external["clean_text"])

before_external_test = len(test_df_external)
test_df_external = test_df_external[~test_df_external["clean_text"].isin(train_text_set)].reset_index(drop=True)
after_external_test = len(test_df_external)

print("Số mẫu test trước khi loại trùng với train:", before_external_test)
print("Số mẫu test sau khi loại trùng với train:", after_external_test)
print("Số mẫu bị loại khỏi test:", before_external_test - after_external_test)

## 20. Train trên dataset 226 và test trên dataset 223

In [ ]:
X_train_ext = train_df_external["clean_text"]
y_train_ext = train_df_external["label"]

X_test_ext = test_df_external["clean_text"]
y_test_ext = test_df_external["label"]

svm_external_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=20000,
        ngram_range=(1, 2),
        min_df=1,
        max_df=0.95,
        sublinear_tf=True
    )),
    ("svm", LinearSVC(
        C=1.0,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ))
])

svm_external_model.fit(X_train_ext, y_train_ext)
y_pred_ext = svm_external_model.predict(X_test_ext)

external_result = {
    "Thực nghiệm": "Train 226 - Test 223",
    "Accuracy": accuracy_score(y_test_ext, y_pred_ext),
    "Precision": precision_score(y_test_ext, y_pred_ext, zero_division=0),
    "Recall": recall_score(y_test_ext, y_pred_ext, zero_division=0),
    "F1-score": f1_score(y_test_ext, y_pred_ext, zero_division=0)
}

print("Kết quả thực nghiệm phụ:")
for key, value in external_result.items():
    print(key, ":", value)

print("\nBáo cáo phân loại:")
print(classification_report(
    y_test_ext,
    y_pred_ext,
    target_names=["Real - Tin thật", "Fake - Tin giả"],
    zero_division=0
))

## 21. Ma trận nhầm lẫn cho thực nghiệm phụ

In [ ]:
cm_ext = confusion_matrix(y_test_ext, y_pred_ext)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm_ext,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Real", "Fake"],
    yticklabels=["Real", "Fake"]
)
plt.title("Confusion Matrix - Train 226, Test 223")
plt.xlabel("Nhãn dự đoán")
plt.ylabel("Nhãn thực tế")
plt.show()

# PHẦN C - Dự đoán tin mới và lưu mô hình

## 22. Hàm dự đoán tin mới

In [ ]:
def predict_fake_news(text, model=best_svm_model):
    cleaned = clean_text(text)
    pred = model.predict([cleaned])[0]
    return label_names[pred]


sample_news = [
    "Bộ Y tế khuyến cáo người dân đeo khẩu trang tại nơi đông người để phòng bệnh.",
    "Một loại thuốc bí mật có thể chữa khỏi mọi bệnh chỉ sau một ngày.",
    "Ngân hàng Nhà nước công bố điều chỉnh lãi suất nhằm ổn định thị trường.",
    "Người ngoài hành tinh xuất hiện tại Hà Nội và bắt tay với lãnh đạo."
]

for news in sample_news:
    print("Tin:", news)
    print("Dự đoán:", predict_fake_news(news))
    print("-" * 80)

## 23. Lưu mô hình SVM tốt nhất

Mô hình được lưu gồm:

- Pipeline TF-IDF + SVM
- Tên nhãn
- Ghi chú tiền xử lý

In [ ]:
MODEL_PATH = "svm_vfnd_merged_fake_news_model.pkl"

joblib.dump({
    "model": best_svm_model,
    "label_names": label_names,
    "note": "Mô hình được train trên dữ liệu VFND đã gộp 2 dataset, xóa trùng và chia train/test."
}, MODEL_PATH)

print("Đã lưu mô hình vào:", MODEL_PATH)

## 24. Load lại mô hình và kiểm thử

In [ ]:
loaded = joblib.load(MODEL_PATH)
loaded_model = loaded["model"]
loaded_label_names = loaded["label_names"]

test_text = "Một thông tin chưa kiểm chứng lan truyền trên mạng xã hội khiến nhiều người hoang mang."
test_cleaned = clean_text(test_text)

pred = loaded_model.predict([test_cleaned])[0]

print("Tin:", test_text)
print("Dự đoán:", loaded_label_names[pred])

# 25. Kết luận mẫu đưa vào báo cáo

Trong bài toán phát hiện tin giả tiếng Việt, nhóm sử dụng hai tập dữ liệu trong bộ VFND là `vn_news_226_tlfr.csv` và `vn_news_223_tdlfr.csv`. Do kích thước dữ liệu tương đối nhỏ, nhóm lựa chọn hướng thực nghiệm chính là gộp hai tập dữ liệu, chuẩn hóa nhãn, tiền xử lý văn bản và loại bỏ các bản ghi trùng lặp trước khi chia train/test. Việc loại bỏ trùng lặp giúp tránh hiện tượng cùng một tin xuất hiện ở cả tập huấn luyện và tập kiểm thử, từ đó làm kết quả đánh giá khách quan hơn.

Sau khi tiền xử lý, dữ liệu văn bản được biểu diễn bằng phương pháp TF-IDF. Mô hình chính được sử dụng là SVM tuyến tính thông qua thuật toán `LinearSVC`. SVM tuyến tính phù hợp với bài toán phân loại văn bản vì dữ liệu TF-IDF thường có số chiều lớn và thưa. Nhóm cũng tiến hành tối ưu tham số bằng `GridSearchCV` để tìm cấu hình tốt hơn cho mô hình.

Kết quả được đánh giá bằng các chỉ số Accuracy, Precision, Recall và F1-score. Ngoài ra, nhóm so sánh SVM với Naive Bayes và Logistic Regression để có cơ sở nhận xét. Bên cạnh thực nghiệm chính, nhóm thực hiện thêm thực nghiệm phụ bằng cách huấn luyện trên một tập dữ liệu và kiểm thử trên tập còn lại nhằm đánh giá khả năng tổng quát hóa của mô hình.